# Nemotron Reasoning Challenge — Kaggle GPU training (L4 × 4)

Trains a LoRA adapter for **NVIDIA-Nemotron-3-Nano-30B-A3B-BF16** on a Kaggle GPU notebook (configured for L4 × 4) and uploads the result to Kaggle as the `nemotron-lora-adapter` Dataset, ready for `kaggle_submission.ipynb` to consume.

## Prerequisites attached via *Add Data*

1. **Competition data** — the `NVIDIA Nemotron Model Reasoning Challenge` (provides `train.csv`, `test.csv`).
2. **Base model** — the Kaggle Model `nvidia/nemotron-3-nano-30b-a3b-bf16` (or another name; resolved automatically).
3. **Scripts dataset** — `<your-username>/nemotron-scripts` (uploaded once from your laptop with `kaggle/scripts/11_upload_scripts_dataset.py --create`).

## Notebook settings

- **Accelerator**: `GPU L4 x 4` (Settings → Accelerator).
- **Internet**: ON for the first run (we may need to `pip install` peft/trl/bitsandbytes if the image doesn't ship them).
- **Persistence**: `Files only` is fine; we write the adapter to `/kaggle/working/`.
- **Add-ons → Secrets**: define `KAGGLE_USERNAME` and `KAGGLE_KEY` so the final cell can publish the adapter as a Kaggle Dataset.

Wall-clock budget on L4 × 4 is roughly 8 – 10 hours; the defaults below are tuned for that envelope.

## Phase 0 — Resolve scripts dir and base model

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").is_dir()
WORK_ROOT = Path("/kaggle/working/project" if IS_KAGGLE else "./project").resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
(WORK_ROOT / "data").mkdir(parents=True, exist_ok=True)
(WORK_ROOT / "data" / "reports").mkdir(parents=True, exist_ok=True)
(WORK_ROOT / "data" / "synthetic").mkdir(parents=True, exist_ok=True)


SENTINEL = "03_train_lora.py"


def _find_scripts_dir() -> Path | None:
    """Locate a directory containing 03_train_lora.py.

    Searches in this order:
      1. /kaggle/input/<any>/scripts/
      2. /kaggle/input/<any>/
      3. Recursive search under /kaggle/input/<any>/ (any nesting)
      4. ./scripts/  and  ./  (running locally next to the notebook)
    """
    quick: list[Path] = []
    deep_roots: list[Path] = []
    if IS_KAGGLE:
        for d in sorted(Path("/kaggle/input").iterdir()):
            if not d.is_dir():
                continue
            quick.append(d / "scripts")
            quick.append(d)
            deep_roots.append(d)
    quick.append(Path.cwd() / "scripts")
    quick.append(Path.cwd())

    for c in quick:
        if (c / SENTINEL).is_file():
            return c.resolve()
    for root in deep_roots:
        try:
            for hit in root.rglob(SENTINEL):
                if hit.is_file():
                    return hit.parent.resolve()
        except OSError:
            continue
    return None


def _diagnose_inputs() -> str:
    if not IS_KAGGLE:
        return "(not running on Kaggle; /kaggle/input does not exist)"
    lines = []
    for d in sorted(Path("/kaggle/input").iterdir()):
        lines.append(f"\n  {d.name}/")
        try:
            entries = sorted(d.iterdir())[:25]
        except OSError as e:
            lines.append(f"    <unreadable: {e}>")
            continue
        for e in entries:
            tag = "/" if e.is_dir() else ""
            lines.append(f"    {e.name}{tag}")
        if len(entries) == 25:
            lines.append("    ... (truncated)")
    return "".join(lines)


SRC_SCRIPTS_DIR = _find_scripts_dir()
if SRC_SCRIPTS_DIR is None:
    diag = _diagnose_inputs()
    raise SystemExit(
        f"Cannot locate '{SENTINEL}' in any attached dataset.\n\n"
        f"Top-level layout under /kaggle/input:{diag}\n\n"
        f"Fix: from your laptop, run\n"
        f"    python3 kaggle/scripts/11_upload_scripts_dataset.py --create\n"
        f"so the dataset is staged with a top-level 'scripts/' folder, then "
        f"detach the current 'nemotron-scripts' from this notebook and "
        f"re-attach the new version via 'Add Data'."
    )

# Mirror scripts into the writable workspace so the .py modules can produce
# files relative to the project root (e.g. data/reports/) without touching the
# read-only /kaggle/input mount.
SCRIPTS_DIR = WORK_ROOT / "scripts"
if SCRIPTS_DIR.exists():
    shutil.rmtree(SCRIPTS_DIR)
shutil.copytree(SRC_SCRIPTS_DIR, SCRIPTS_DIR)
for p in (str(WORK_ROOT), str(SCRIPTS_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"SRC_SCRIPTS_DIR = {SRC_SCRIPTS_DIR}")
print(f"WORK_ROOT       = {WORK_ROOT}")
print(f"SCRIPTS_DIR     = {SCRIPTS_DIR}")

# Resolve base model.
import importlib.util
_spec = importlib.util.spec_from_file_location(
    "_paths", SCRIPTS_DIR / "kaggle_nemotron_paths.py"
)
_paths = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_paths)  # type: ignore[union-attr]
MODEL_PATH_LOCAL = _paths.find_kaggle_competition_nemotron_dir()
if MODEL_PATH_LOCAL is None:
    raise SystemExit(
        "Nemotron base model not found under /kaggle/input. Attach the Kaggle "
        "Model 'nvidia/nemotron-3-nano-30b-a3b-bf16' via Add Data → Models."
    )
print(f"BASE_MODEL_DIR  = {MODEL_PATH_LOCAL}")

## Phase 0b — Locate competition data and stage into project workspace

In [ ]:
def _find_competition_csv(name: str) -> Path | None:
    root = Path("/kaggle/input") if IS_KAGGLE else Path("data")
    if not root.is_dir():
        return None
    # Prefer the direct child of the competition dataset folder.
    for top in sorted(root.iterdir()):
        if top.is_dir() and (top / name).is_file():
            return top / name
    # Fall back to a recursive search.
    for hit in root.rglob(name):
        if hit.is_file():
            return hit
    return None


TRAIN_CSV_SRC = _find_competition_csv("train.csv")
TEST_CSV_SRC = _find_competition_csv("test.csv")
if TRAIN_CSV_SRC is None or TEST_CSV_SRC is None:
    raise SystemExit(
        "Could not find train.csv / test.csv under /kaggle/input. "
        "Attach the competition data via 'Add Data → Competitions'."
    )

DATA_DIR = WORK_ROOT / "data"
for src in (TRAIN_CSV_SRC, TEST_CSV_SRC):
    dst = DATA_DIR / src.name
    if not dst.is_file():
        shutil.copy2(src, dst)
    print(f"  staged {src} -> {dst}")

## Phase 1 — Install training dependencies

The Kaggle GPU image already ships PyTorch + CUDA + transformers. We add `peft`, `trl`, `accelerate`, `bitsandbytes` (the last one is only needed if you flip `USE_BF16_FULL = False` to enable 4-bit; harmless to install).

In [ ]:
import importlib, socket
from importlib.metadata import version as _pkg_version, PackageNotFoundError

_FLOORS = {
    "transformers": "4.45.0",
    "peft": "0.13.0",
    "trl": "0.11.0",
    "accelerate": "0.34.0",
    "bitsandbytes": "0.43.0",
    "datasets": "3.0.0",
}

# Offline install: scan a few well-known mount paths for a folder containing
# the wheels we need. Built once with kaggle/scripts/12_build_nemotron_wheels_dataset.py.
_WHEEL_DIR_CANDIDATES = [
    "/kaggle/input/nemotron-wheels",
    "/kaggle/input/training-wheels",
]
TRAINING_WHEELS_DIR = next(
    (d for d in _WHEEL_DIR_CANDIDATES if Path(d).is_dir() and any(Path(d).glob("*.whl"))),
    "/kaggle/input/nemotron-wheels",
)


def _parse(v: str) -> tuple[int, ...]:
    parts = []
    for chunk in v.split("+", 1)[0].split("."):
        try:
            parts.append(int(chunk))
        except ValueError:
            break
    return tuple(parts)


def _have_internet() -> bool:
    try:
        socket.create_connection(("pypi.org", 443), timeout=3).close()
        return True
    except OSError:
        return False


to_install: list[str] = []
for pkg, floor in _FLOORS.items():
    try:
        cur = _pkg_version(pkg)
    except PackageNotFoundError:
        to_install.append(f"{pkg}>={floor}")
        continue
    if _parse(cur) < _parse(floor):
        print(f"  {pkg} {cur} < {floor} -> needs upgrade")
        to_install.append(f"{pkg}>={floor}")
    else:
        print(f"  {pkg} {cur} OK (>= {floor})")

if not to_install:
    print("All training deps already at required versions.")
else:
    online = _have_internet()
    have_wheels = Path(TRAINING_WHEELS_DIR).is_dir() and any(
        Path(TRAINING_WHEELS_DIR).glob("*.whl")
    )
    if online:
        print(f"Online -> pip installing from PyPI: {to_install}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *to_install]
        )
    elif have_wheels:
        print(f"Offline -> installing from {TRAINING_WHEELS_DIR}: {to_install}")
        # --no-deps: the wheels dataset also ships vLLM / xformers / triton
        # wheels (used by 04_evaluate.py, not by training). Letting pip
        # resolve dependencies from --find-links can silently pull those
        # in as transitives of trl/transformers and downgrade triton +
        # install xformers 0.0.27 / vllm 0.6.3, which were built against
        # numpy 1.x. Combined with Kaggle's numpy 2.x base image that
        # triggers
        #   ValueError: numpy.dtype size changed, may indicate binary
        #   incompatibility. Expected 96 from C header, got 88 from PyObject
        # on `import vllm` / `import xformers`. Training only needs the
        # listed floors; the floor packages' transitive deps are already
        # present in Kaggle's image, so --no-deps is safe here.
        subprocess.check_call(
            [
                sys.executable, "-m", "pip", "install",
                "--no-index", "--find-links", TRAINING_WHEELS_DIR,
                "--no-deps",
                "--upgrade", *to_install,
            ]
        )
    else:
        raise SystemExit(
            "Internet is off AND no wheels Kaggle Dataset is attached.\n"
            "Either:\n"
            "  (A) Settings -> Internet: ON, re-run this cell, OR\n"
            "  (B) From your laptop run\n"
            "        python3 kaggle/scripts/12_build_nemotron_wheels_dataset.py --create\n"
            "      then 'Add Data -> Your Datasets -> nemotron-wheels' and re-run.\n"
            f"Missing: {to_install}"
        )
    importlib.invalidate_caches()


# Pin torch to 2.4.0+cu121 so the prebuilt CUDA wheel stack in nemotron-wheels
# stays internally consistent.
#
# Kaggle's Python 3.12 GPU image has been bumped to torch 2.10. The
# nemotron-wheels Kaggle Dataset ships mamba_ssm / causal_conv1d / xformers
# 0.0.27.post2 / vllm 0.6.3 / triton 3.1 wheels all built against torch 2.4 +
# cu121 + cp312, plus numpy-1.x ABI. If Kaggle's torch isn't 2.4.x, the next
# guard (_ensure_mamba_deps) bails with
#   "No mamba_ssm wheel for torch X.Y in /kaggle/input/nemotron-wheels"
# because no matching wheel exists for the new torch. Pin torch back to 2.4
# before that check fires.
def _pin_torch_2_4() -> None:
    target_prefix = "2.4."
    torch_loaded = "torch" in sys.modules
    try:
        cur = _pkg_version("torch")
    except PackageNotFoundError:
        cur = None
    if cur and cur.startswith(target_prefix):
        print(f"torch {cur} -> already on the 2.4 line, no downgrade needed")
        return

    print(f"torch {cur} -> pinning to 2.4.0+cu121 (matches prebuilt CUDA wheels)")

    wheels_dir = Path(TRAINING_WHEELS_DIR)
    local_torch = sorted(wheels_dir.glob("torch-2.4.0*cu121*cp312*.whl"))
    local_tv = sorted(wheels_dir.glob("torchvision-0.19.0*cu121*cp312*.whl"))

    install_args: list[str] = [
        sys.executable, "-m", "pip", "install", "-q",
        # --no-deps: torch's metadata pulls numpy. We re-pin numpy==1.26.4
        # immediately after, but skipping deps here keeps the install
        # narrow and predictable.
        "--no-deps", "--force-reinstall",
    ]
    if local_torch and local_tv:
        print(f"  using local wheels: {local_torch[-1].name}, {local_tv[-1].name}")
        install_args += [
            "--no-index", "--find-links", str(wheels_dir),
            "torch==2.4.0+cu121", "torchvision==0.19.0+cu121",
        ]
    elif _have_internet():
        print("  no local torch wheels; downloading from pytorch.org/whl/cu121")
        install_args += [
            "--index-url", "https://download.pytorch.org/whl/cu121",
            "torch==2.4.0+cu121", "torchvision==0.19.0+cu121",
        ]
    else:
        raise SystemExit(
            "Cannot pin torch to 2.4.0+cu121: Kaggle internet is OFF AND no "
            f"local torch/torchvision wheels in {wheels_dir}.\n"
            "Fix one of:\n"
            "  (A) Settings -> Internet: ON in this notebook, re-run this cell.\n"
            "  (B) From your laptop run\n"
            "        python3 kaggle/scripts/12_build_nemotron_wheels_dataset.py\n"
            "      (which now downloads torch 2.4.0+cu121 + torchvision\n"
            "      0.19.0+cu121 from pytorch.org/whl/cu121), bump the\n"
            "      nemotron-wheels Kaggle Dataset version, then re-attach it\n"
            "      to this notebook.\n"
            f"Current torch: {cur}"
        )

    subprocess.check_call(install_args)

    # If torch was already imported into this kernel, the in-memory C
    # extension is the OLD one (2.10) -- we cannot swap it under a live
    # interpreter. Force a restart instead of silently shipping a broken
    # mixed-version stack.
    if torch_loaded:
        print(
            "\nIMPORTANT: torch was already imported in this kernel before the\n"
            "downgrade. Hot-swapping a live torch is not safe.\n"
            "  -> Restart the kernel (Run -> Restart Kernel & Run All) so the\n"
            "     re-import picks up torch 2.4.0+cu121.\n"
        )
        raise SystemExit("Restart the kernel to apply the torch 2.4.0+cu121 pin.")

    for _mod in [
        m for m in list(sys.modules)
        if m == "torch" or m.startswith("torch.")
        or m == "torchvision" or m.startswith("torchvision.")
    ]:
        sys.modules.pop(_mod, None)
    importlib.invalidate_caches()


# Pin numpy<2 so the prebuilt CUDA wheels in nemotron-wheels stay importable.
#
# The xformers 0.0.27 / vllm 0.6.3 / triton 3.1 / torch 2.4 wheels staged in
# the wheels dataset were all built against numpy 1.x. Kaggle's Python 3.12
# image now ships numpy 2.x, and any C-extension import on top of that combo
# explodes with
#   ValueError: numpy.dtype size changed, may indicate binary incompatibility.
#                Expected 96 from C header, got 88 from PyObject
# Force-reinstall numpy==1.26.4 (last stable 1.x) with --no-deps so pip
# doesn't drag a numpy-2-rebuilt scipy/pandas back in alongside it.
def _pin_numpy_v1() -> None:
    try:
        import numpy as _np
    except ImportError:
        _np = None  # type: ignore[assignment]
    cur = getattr(_np, "__version__", "0.0.0")
    if cur.startswith("1."):
        print(f"numpy {cur} -> already 1.x, no downgrade needed")
        return
    print(f"numpy {cur} -> pinning to 1.26.4 (numpy-1.x ABI for prebuilt wheels)")
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--force-reinstall", "--no-deps", "numpy==1.26.4",
        ]
    )
    importlib.invalidate_caches()
    # Drop any already-imported numpy so the next `import numpy` picks up 1.x.
    for _mod in [m for m in list(sys.modules) if m == "numpy" or m.startswith("numpy.")]:
        sys.modules.pop(_mod, None)


# Order matters: pin torch BEFORE _ensure_mamba_deps (which picks wheels
# based on torch.__version__). Re-pin numpy after the torch reinstall in
# case --force-reinstall ever pulled a numpy-2 build back in.
_pin_torch_2_4()
_pin_numpy_v1()


def _ensure_mamba_deps() -> None:
    """Install mamba-ssm + causal-conv1d wheels matching Kaggle's torch.

    Nemotron-3 uses Mamba layers, so 03_train_lora.py fails to load the
    base model without these. PyPI ships only sdists, so the nemotron-wheels
    dataset (v3+) bundles prebuilt CUDA wheels for torch 2.4 (cu12, cp312,
    cxx11abi=FALSE). We pick whichever matches the (now-pinned) torch.
    """
    try:
        import torch
    except ImportError:
        print("torch not importable yet; skipping mamba install.")
        return

    have_mamba = True
    have_conv = True
    try:
        import mamba_ssm  # noqa: F401
    except ImportError:
        have_mamba = False
    try:
        import causal_conv1d  # noqa: F401
    except ImportError:
        have_conv = False
    if have_mamba and have_conv:
        print("mamba-ssm + causal-conv1d already importable.")
        return

    import re
    import shutil
    import tempfile

    torch_minor = ".".join(torch.__version__.split("+", 1)[0].split(".")[:2])
    print(f"detected torch {torch.__version__} (matching minor: {torch_minor})")

    wheels_dir = Path(TRAINING_WHEELS_DIR)

    # Kaggle strips '+' from filenames on upload, so a wheel uploaded as
    # 'mamba_ssm-2.2.4+cu122torch2.4cxx11abiFALSE-...whl' is stored as
    # 'mamba_ssm-2.2.4cu122torch2.4cxx11abiFALSE-...whl'. _ensure_plus() below
    # re-injects the '+' before pip-install because pip needs the PEP 440
    # local-version form.
    #
    # Filename convention (from state-spaces/mamba GitHub releases):
    #   {prefix}-{version}+cu{NNN}torch{X.Y}cxx11abi{TRUE|FALSE}-cp312-cp312-linux_x86_64.whl
    # where cu{NNN} is cu118 or cu122 (NOT 'cu12' -- earlier versions of
    # _pick() hard-coded that literal and never matched anything in the
    # dataset). We anchor on the torch{X.Y} segment and let cu{NNN} float.
    torch_full = torch.__version__.split("+", 1)[0]    # e.g. "2.4.0"
    def _pick(prefix: str) -> Path | None:
        patterns = [
            # Most specific: ABI=FALSE (the bf16-safe build) + matching torch minor.
            f"{prefix}-*torch{torch_minor}cxx11abiFALSE-cp312-cp312-linux_x86_64.whl",
            # Any cxx11abi variant for the matching torch minor.
            f"{prefix}-*torch{torch_minor}*-cp312-cp312-linux_x86_64.whl",
            # Some uploaders include the patch ('torch2.4.0' instead of 'torch2.4').
            f"{prefix}-*torch{torch_full}*-cp312-cp312-linux_x86_64.whl",
            # Trust the dataset: any cp312 linux wheel for this prefix. If
            # the uploader put a wrong-torch wheel here, the import below
            # this cell will fail with a clearer ABI error than this
            # filename-pattern bail-out would.
            f"{prefix}-*-cp312-cp312-linux_x86_64.whl",
        ]
        for pat in patterns:
            cands = sorted(wheels_dir.glob(pat))
            if cands:
                # Prefer cu121 (matches our torch 2.4.0+cu121 pin), then cu122
                # (binary-compatible with cu121 at the driver level), then any
                # other cu*, then no-cuda last.
                def _rank(p: Path) -> tuple[int, str]:
                    n = p.name
                    if "cu121" in n: r = 0
                    elif "cu122" in n: r = 1
                    elif "cu12" in n: r = 2
                    elif "cu11" in n: r = 3
                    else: r = 4
                    return (r, n)
                cands.sort(key=_rank)
                print(f"  picked {prefix}: {cands[0].name} (pattern: {pat})")
                return cands[0]
        return None

    def _ensure_plus(src: Path) -> Path:
        """Return a path whose filename contains '+cu...' (PEP 440 local version)."""
        if "+cu" in src.name:
            return src
        new_name = re.sub(r"(cu\d+torch)", r"+\1", src.name, count=1)
        if new_name == src.name:
            return src
        dst = Path(tempfile.gettempdir()) / new_name
        if not dst.exists():
            try:
                dst.symlink_to(src)
            except OSError:
                shutil.copy(src, dst)
        return dst

    def _diagnose_missing(prefix: str) -> str:
        """Return a string listing every wheel in wheels_dir starting with <prefix>."""
        hits = sorted(p.name for p in wheels_dir.glob(f"{prefix}*"))
        if not hits:
            return f"  (no files matching '{prefix}*' in {wheels_dir})"
        return "\n".join(f"  {h}" for h in hits)

    needed_src: list[Path] = []
    if not have_mamba:
        w = _pick("mamba_ssm")
        if w is None:
            raise SystemExit(
                f"No mamba_ssm wheel matching torch {torch_minor} (or any "
                f"cp312 fallback) in {wheels_dir}.\n"
                f"Files starting with 'mamba_ssm' that ARE present:\n"
                f"{_diagnose_missing('mamba_ssm')}\n\n"
                "Expected filename convention (from state-spaces/mamba releases):\n"
                "  mamba_ssm-<ver>+cu{118|122}torch<minor>cxx11abiFALSE-"
                "cp312-cp312-linux_x86_64.whl\n"
                "Two remedies:\n"
                "  (A) Add the matching wheel to the nemotron-wheels Kaggle\n"
                "      Dataset (download from state-spaces/mamba releases) and\n"
                "      bump the dataset version.\n"
                "  (B) If torch is somehow NOT 2.4.x here, restart the kernel\n"
                "      and Run All so _pin_torch_2_4() at the top of this cell\n"
                "      can downgrade torch before this check fires."
            )
        needed_src.append(w)
    if not have_conv:
        w = _pick("causal_conv1d")
        if w is None:
            raise SystemExit(
                f"No causal_conv1d wheel matching torch {torch_minor} (or any "
                f"cp312 fallback) in {wheels_dir}.\n"
                f"Files starting with 'causal_conv1d' that ARE present:\n"
                f"{_diagnose_missing('causal_conv1d')}\n\n"
                "Expected filename convention (from Dao-AILab/causal-conv1d releases):\n"
                "  causal_conv1d-<ver>+cu{118|122}torch<minor>cxx11abiFALSE-"
                "cp312-cp312-linux_x86_64.whl\n"
                "Same remedies as the mamba_ssm message above."
            )
        needed_src.append(w)

    needed = [_ensure_plus(w) for w in needed_src]
    print("Installing:", [w.name for w in needed])
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install",
            "--no-index", "--no-deps",
            *[str(w) for w in needed],
        ]
    )


_ensure_mamba_deps()


_pin_numpy_v1()

In [ ]:
# Sanity-import + numpy ABI guard.
#
# The vLLM / xformers / triton wheels in nemotron-wheels were built against
# numpy 1.x. If numpy-2.x has snuck back in (e.g. wheels dataset re-attached
# without re-running the install cell, or a later `pip install` upgraded
# numpy as a transitive), every CUDA-extension import below this cell will
# die with
#   ValueError: numpy.dtype size changed, may indicate binary incompatibility.
#                Expected 96 from C header, got 88 from PyObject
# Catch that here loudly instead of inside the 30B model load.
import importlib as _importlib  # noqa: E402
import numpy as _np  # noqa: E402

print(f"numpy        {_np.__version__:>12s} -> {_np.__file__}")
if _np.__version__.startswith("2."):
    raise SystemExit(
        f"ABI guard: numpy {_np.__version__} is numpy-2.x but the prebuilt CUDA "
        f"wheels in {TRAINING_WHEELS_DIR} were built against numpy 1.x. Re-run "
        "the install cell above (it pins numpy==1.26.4), or run:\n"
        "  !pip install --force-reinstall --no-deps numpy==1.26.4"
    )

# torch ABI guard: every CUDA C-extension wheel staged in nemotron-wheels
# (mamba_ssm, causal_conv1d, xformers 0.0.27.post2, vllm 0.6.3, triton 3.1)
# was built for torch 2.4 + cu121. If we somehow ran past _pin_torch_2_4()
# with a non-2.4 torch (kernel not restarted after the pin, fresh install
# clobbered the downgrade, etc.), every subsequent import below will explode
# at runtime in confusing ways. Catch it here.
import torch as _torch  # noqa: E402

print(f"torch        {_torch.__version__:>12s} -> {_torch.__file__}")
if not _torch.__version__.startswith("2.4."):
    raise SystemExit(
        f"ABI guard: torch {_torch.__version__} is NOT 2.4.x but the prebuilt CUDA "
        f"wheels in {TRAINING_WHEELS_DIR} were built for torch 2.4 + cu121.\n"
        "Re-run the install cell above (it calls _pin_torch_2_4(), which "
        "downgrades to 2.4.0+cu121) AND restart the kernel afterwards so the "
        "downgrade takes effect."
    )

for _name in (
    "transformers", "peft", "trl", "accelerate",
    "bitsandbytes", "datasets", "mamba_ssm", "causal_conv1d",
):
    try:
        _m = _importlib.import_module(_name)
        _v = getattr(_m, "__version__", "?")
        _f = getattr(_m, "__file__", "?")
        print(f"{_name:<12s} {_v:>12s} -> {_f}")
    except Exception as _e:
        print(f"{_name:<12s} NOT importable ({type(_e).__name__}: {_e})")


## Phase 2 — Configuration (L4 × 4)

Defaults assume four L4 GPUs (96 GB total VRAM). The 30B base in bf16 needs ~60 GB; LoRA + activations fit in the remaining headroom with `device_map="auto"`.

In [ ]:
import torch

GPU_COUNT = torch.cuda.device_count()
print(f"GPU_COUNT = {GPU_COUNT}")
for i in range(GPU_COUNT):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name} ({p.total_memory/1e9:.1f} GB)")

USE_BF16_FULL = True
TRAIN_MAX_SEQ = 4096  # bumped from 2048 (teacher-distilled CoTs are longer)
LORA_TARGET_MODE = "kaggle_nemotron"
LORA_ALPHA = 64       # bumped from 32 (with rsLoRA at r=32 -> effective scaling ~11.3)
TRAIN_BATCH = 1
GRAD_ACCUM = 16
NUM_EPOCHS = 2.0      # bumped from 1.0 (larger data pool after Phase 1)
LR = 1e-4
SKIP_COT = True       # no API key in Kaggle env by default; uses template + solver CoTs
COMPLETION_ONLY = True  # mask system+user, train only on assistant tokens
SYNTHETIC_PER_KIND = 2500  # default per-kind; per-kind overrides below
# Phase 1.1 per-kind budgets, tuned to ACTUAL test distribution (winner ref):
# bit_manipulation + cipher are real test categories; our algebraic/sequence
# families do NOT appear in the test set, so keep them minimal.
SYNTHETIC_BIT = 4000
SYNTHETIC_CIPHER = 4000
SYNTHETIC_ALGEBRAIC = 500
SYNTHETIC_SEQUENCE = 500
RUN_PSEUDOLABEL = True

# Phase 3: GRPO Stage 2 (warm-start from SFT adapter, optimize answer correctness)
RUN_GRPO = True             # set False to skip the GRPO stage
GRPO_EPOCHS = 1.0
GRPO_LR = 5e-6
GRPO_LIMIT = 600            # 0 = all training rows
GRPO_NUM_GENERATIONS = 4    # rollouts per prompt (L4 24GB: 2-4)
GRPO_MAX_NEW_TOKENS = 1024
GRPO_MAX_PROMPT_LEN = 1024
GRPO_BATCH = 1
GRPO_GRAD_ACCUM = 4

if GPU_COUNT >= 4:
    per_gpu = max(int(torch.cuda.get_device_properties(0).total_memory / 1e9) - 2, 16)
    TRAIN_MAX_MEMORY_JSON = json.dumps(
        {**{str(i): f"{per_gpu}GB" for i in range(GPU_COUNT)}, "cpu": "32GB"}
    )
else:
    TRAIN_MAX_MEMORY_JSON = None

print(f"USE_BF16_FULL        = {USE_BF16_FULL}")
print(f"TRAIN_MAX_MEMORY_JSON = {TRAIN_MAX_MEMORY_JSON}")

## Phase 3 — EDA (sanity check)

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(SCRIPTS_DIR / "01_eda.py"),
        "--data-dir", str(DATA_DIR),
        "--report-dir", str(DATA_DIR / "reports"),
        "--tokenizer-model", str(MODEL_PATH_LOCAL),
    ],
    check=True,
    cwd=str(WORK_ROOT),
)

## Phase 3b — Solver pseudo-labels on `test.csv`

In [ ]:
if RUN_PSEUDOLABEL:
    subprocess.run(
        [
            sys.executable,
            str(SCRIPTS_DIR / "02b_pseudolabel_test.py"),
            "--test-csv", str(DATA_DIR / "test.csv"),
            "--output", str(DATA_DIR / "pseudo_test.jsonl"),
            "--report-dir", str(DATA_DIR / "reports"),
        ],
        check=True,
        cwd=str(WORK_ROOT),
    )
else:
    print("Skipping pseudo-labeling (RUN_PSEUDOLABEL=False).")

## Phase 4 — Prepare SFT data

In [ ]:
prep_env = os.environ.copy()
prep_env["PYTHONUNBUFFERED"] = "1"

cot_args = ["--skip-cot"] if SKIP_COT else ["--cot-backend", "openai", "--cot-model", "gpt-4o"]
pseudo_args: list[str] = []
pseudo_path = DATA_DIR / "pseudo_test.jsonl"
if pseudo_path.is_file():
    pseudo_args = ["--pseudo-label-file", str(pseudo_path)]
    print(f"Including pseudo-labels from {pseudo_path}")

cmd = [
    sys.executable,
    str(SCRIPTS_DIR / "02_prepare_data.py"),
    "--data-dir", str(DATA_DIR),
    "--synthetic-dir", str(DATA_DIR / "synthetic"),
    "--output", str(DATA_DIR / "train_sft.jsonl"),
    "--tokenizer-model", str(MODEL_PATH_LOCAL),
    "--synthetic-per-kind", str(SYNTHETIC_PER_KIND),
    "--bit", str(SYNTHETIC_BIT),
    "--cipher", str(SYNTHETIC_CIPHER),
    "--algebraic", str(SYNTHETIC_ALGEBRAIC),
    "--sequence", str(SYNTHETIC_SEQUENCE),
    "--max-tokens-per-example", str(TRAIN_MAX_SEQ - 256),
    "--max-per-type", "6000",
    "--curriculum",  # easy -> hard within one epoch
    *cot_args,
    *pseudo_args,
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=prep_env, cwd=str(WORK_ROOT))

## Phase 5 — LoRA training (bf16, sharded across GPUs)

In [ ]:
import gc

ADAPTER_DIR = WORK_ROOT / "lora_adapter"
CKPT_DIR = WORK_ROOT / "lora_output"

train_env = os.environ.copy()
train_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
train_env["TOKENIZERS_PARALLELISM"] = "false"
train_env["NEMOTRON_KAGGLE_PATCHES"] = "0"
train_env["PYTHONUNBUFFERED"] = "1"

# Phase 2 SFT v2: rank=32 cap, alpha=64, epochs=2, max_seq=4096,
# completion-only, NEFTune 5.0 (script default), rsLoRA on (default).
cmd = [
    sys.executable,
    str(SCRIPTS_DIR / "03_train_lora.py"),
    "--data-path", str(DATA_DIR / "train_sft.jsonl"),
    "--output-dir", str(ADAPTER_DIR),
    "--checkpoint-dir", str(CKPT_DIR),
    "--model-path", str(MODEL_PATH_LOCAL),
    "--lora-target-mode", LORA_TARGET_MODE,
    "--lora-r", "32",         # competition rank cap
    "--lora-alpha", str(LORA_ALPHA),
    "--batch-size", str(TRAIN_BATCH),
    "--grad-accum", str(GRAD_ACCUM),
    "--epochs", str(NUM_EPOCHS),
    "--lr", str(LR),
    "--max-seq-length", str(TRAIN_MAX_SEQ),
    "--force-peft",
    "--no-nemotron-kaggle-patches",
    "--dataloader-workers", "0",
    # Compute loss only on the assistant turn (TRL DataCollatorForCompletionOnlyLM).
    # Masks system + user tokens so the model only learns to generate the
    # reasoning + boxed answer, not to reproduce the prompt. Typically +1-3pp
    # on small SFT sets. Requires no --packing (incompatible in 03_train_lora.py).
    "--completion-only",
]
if USE_BF16_FULL:
    cmd.append("--no-quant")
if TRAIN_MAX_MEMORY_JSON:
    cmd += ["--max-memory-json", TRAIN_MAX_MEMORY_JSON]

gc.collect()
print(" ".join(cmd), flush=True)

proc = subprocess.Popen(
    cmd,
    env=train_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=str(WORK_ROOT),
)
for line in proc.stdout:  # type: ignore[union-attr]
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\nAdapter contents:")
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

## Phase 5b — GRPO Stage 2 (optional; warm-started from the SFT adapter)

GRPO uses the gold answers in `train.csv` as a reward signal — no teacher API.
Docstring claim in `08_grpo.py`: takes a 0.59 SFT adapter to 0.88+.

Settings are tuned for L4 × 4 (24 GB per device; the script's `_load_model` reads
device 0). Drop `GRPO_NUM_GENERATIONS` to 2 first if you OOM, then
`GRPO_MAX_NEW_TOKENS` to 768.

In [ ]:
if RUN_GRPO:
    GRPO_ADAPTER_DIR = WORK_ROOT / "lora_adapter_grpo"
    GRPO_CKPT_DIR = WORK_ROOT / "grpo_output"

    grpo_env = os.environ.copy()
    grpo_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    grpo_env["TOKENIZERS_PARALLELISM"] = "false"
    grpo_env["PYTHONUNBUFFERED"] = "1"

    grpo_cmd = [
        sys.executable,
        str(SCRIPTS_DIR / "08_grpo.py"),
        "--train-csv", str(DATA_DIR / "train.csv"),
        "--sft-adapter", str(ADAPTER_DIR),
        "--base-model", str(MODEL_PATH_LOCAL),
        "--output-dir", str(GRPO_ADAPTER_DIR),
        "--checkpoint-dir", str(GRPO_CKPT_DIR),
        "--lora-r", "32",
        "--lora-alpha", str(LORA_ALPHA),
        "--epochs", str(GRPO_EPOCHS),
        "--lr", str(GRPO_LR),
        "--batch-size", str(GRPO_BATCH),
        "--grad-accum", str(GRPO_GRAD_ACCUM),
        "--num-generations", str(GRPO_NUM_GENERATIONS),
        "--max-new-tokens", str(GRPO_MAX_NEW_TOKENS),
        "--max-prompt-len", str(GRPO_MAX_PROMPT_LEN),
        "--limit", str(GRPO_LIMIT),
    ]

    print(" ".join(grpo_cmd), flush=True)
    gc.collect()

    proc = subprocess.Popen(
        grpo_cmd,
        env=grpo_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=str(WORK_ROOT),
    )
    for line in proc.stdout:  # type: ignore[union-attr]
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        # Do not crash the whole notebook on GRPO failure: keep the SFT adapter
        # as the publishable artifact. Common failure modes: OOM on rollouts
        # (cut --num-generations to 2 or --max-new-tokens to 768) and TRL
        # version skew (already defended against in 08_grpo.py).
        print(
            f"\n[WARN] GRPO exited with code {rc}. Keeping SFT adapter at "
            f"{ADAPTER_DIR}. Set RUN_GRPO=False to skip on next run."
        )
    else:
        # Point downstream phases at the GRPO adapter.
        ADAPTER_DIR = GRPO_ADAPTER_DIR
        print(f"\n[Phase 5b] GRPO finished. Publishing GRPO adapter at {ADAPTER_DIR}.")
        for f in sorted(ADAPTER_DIR.iterdir()):
            print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
else:
    print("RUN_GRPO=False — skipping GRPO Stage 2.")

## Phase 6 — Stage adapter into `/kaggle/working/lora_adapter`

We mirror the adapter files into `/kaggle/working/` so they appear in the notebook's *Output* tab — useful as a UI fallback if the scripted upload below fails (you can right-click the folder → *Save as Dataset*).

In [ ]:
OUTPUT_ADAPTER = Path("/kaggle/working/lora_adapter") if IS_KAGGLE else WORK_ROOT / "_output_adapter"
if OUTPUT_ADAPTER.exists():
    shutil.rmtree(OUTPUT_ADAPTER)
OUTPUT_ADAPTER.mkdir(parents=True)
for src in ADAPTER_DIR.iterdir():
    if src.is_file():
        shutil.copy2(src, OUTPUT_ADAPTER / src.name)
print(f"Mirrored adapter to: {OUTPUT_ADAPTER}")
for f in sorted(OUTPUT_ADAPTER.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

## Phase 7 — Publish adapter as `nemotron-lora-adapter` Kaggle Dataset

Set `FIRST_TIME = True` for the very first publish; flip to `False` for subsequent retraining runs (creates a new version instead).

Reads `KAGGLE_USERNAME` / `KAGGLE_KEY` from **Kaggle Secrets** (Add-ons → Secrets). Falls back to `~/.kaggle/kaggle.json` if present.

In [ ]:
KAGGLE_DATASET_ID = "sebmontreal/nemotron-lora-adapter"
KAGGLE_DATASET_TITLE = "Nemotron LoRA Adapter (Reasoning Challenge)"
FIRST_TIME = True
VERSION_NOTES = "v1 - initial upload (Kaggle L4x4)"

if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore[import-not-found]
        _secrets = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = _secrets.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = _secrets.get_secret("KAGGLE_KEY")
        print("Loaded Kaggle credentials from Kaggle Secrets.")
    except Exception as e:
        if not (Path.home() / ".kaggle" / "kaggle.json").is_file():
            raise RuntimeError(
                "No Kaggle credentials. Define KAGGLE_USERNAME and KAGGLE_KEY "
                "in 'Add-ons → Secrets', or upload kaggle.json to "
                "~/.kaggle/kaggle.json before running this cell."
            ) from e
        print("Using ~/.kaggle/kaggle.json (Kaggle Secrets unavailable).")

def _import_kaggle_cli() -> bool:
    """Try to import the kaggle package. Importing it triggers
    api.authenticate() which makes a network call; if Internet is OFF in
    the Kaggle notebook, that raises ConnectionError. Treat any failure
    here (ImportError, ConnectionError, RuntimeError, ...) as 'unusable'.
    """
    try:
        import kaggle  # type: ignore[import-not-found]  # noqa: F401
        return True
    except ImportError:
        return False
    except Exception as exc:  # noqa: BLE001
        print(f"'kaggle' is installed but not usable: {type(exc).__name__}: {exc}")
        return False

have_kaggle_cli = _import_kaggle_cli()

if not have_kaggle_cli:
    install_args: list[str] = [sys.executable, "-m", "pip", "install", "-q", "kaggle"]
    if Path(TRAINING_WHEELS_DIR).is_dir() and any(Path(TRAINING_WHEELS_DIR).glob("*.whl")):
        install_args = [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", TRAINING_WHEELS_DIR, "kaggle",
        ]
    try:
        subprocess.check_call(install_args)
        have_kaggle_cli = _import_kaggle_cli()
    except subprocess.CalledProcessError:
        have_kaggle_cli = False

if not have_kaggle_cli:
    print(
        "Skipping scripted upload (kaggle package missing or no internet).\n"
        f"FALLBACK: the trained adapter is mirrored to {OUTPUT_ADAPTER}.\n"
        "After this notebook finishes, click 'Save Version -> Save & Run All',\n"
        "open the run's Output tab, and use 'New Dataset' on the lora_adapter\n"
        f"folder. Name it '{KAGGLE_DATASET_ID.split('/', 1)[1]}'.\n"
        "Or: enable Internet (Settings -> Internet -> On, requires phone-verified\n"
        "Kaggle account) and re-run this cell to use the scripted upload."
    )

if have_kaggle_cli:
    cmd = [
        sys.executable,
        str(SCRIPTS_DIR / "08_upload_adapter_dataset.py"),
        "--adapter-dir", str(ADAPTER_DIR),
        "--dataset-id", KAGGLE_DATASET_ID,
        "--title", KAGGLE_DATASET_TITLE,
    ]
    if FIRST_TIME:
        cmd.append("--first-time")
    else:
        cmd += ["--version-notes", VERSION_NOTES]

    print(" ".join(cmd))
    subprocess.run(cmd, check=True, cwd=str(WORK_ROOT))
    print(
        "\nDone. Open: https://www.kaggle.com/datasets/" + KAGGLE_DATASET_ID
    )